# Calculus and Optimizations

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import seaborn as sns
from ipywidgets import FloatSlider, interact


### Concept 1: Numerical Derivatives and Slopes
### What is a Derivative?

Imagine you are driving a car and you press the gas pedal down by just **1 millimeter** for a fraction of a second.

**The derivative answers one core question:**

> *"If I make an infinitely small nudge to my input, how much does my output change?"*

* **Slope / Derivative > 0:** If I increase the input, the output goes **up**.
* **Slope / Derivative < 0:** If I increase the input, the output goes **down**.
* **Slope / Derivative = 0:** You are at a flat spot (a peak, a valley, or a saddle point).

In machine learning, the output is almost always your **Loss** (error), and the inputs are your **Weights**. The derivative tells you: *"Should I increase or decrease this weight to reduce the loss?"*


### The Formulas (Finite Differences)

To compute a derivative with code when you don't want to do pen-and-paper calculus, we nudge the input $x$ by a tiny amount $\epsilon$ (epsilon, like $10^{-5}$) and measure the change in $f(x)$.

**1. Forward Difference (Intuitive, but less accurate):**

$$f'(x) \approx \frac{f(x + \epsilon) - f(x)}{\epsilon}$$

**2. Centered Difference (Symmetric, much more accurate):**

Nudge forward by half a step, and backward by half a step:

$$f'(x) \approx \frac{f(x + \epsilon) - f(x - \epsilon)}{2\epsilon}$$

#### How to read it out loud:

> *"The derivative f-prime of x is approximately f of x plus epsilon, minus f of x minus epsilon, divided by two epsilon."*

#### Breakdown of every piece:

* **$f'(x)$**: The instantaneous rate of change (slope) of function $f$ at position $x$.
* **$\epsilon$** (*epsilon*): A tiny nudge value (e.g., $10^{-5}$). It must be small enough to approximate zero, but not so tiny that floating-point rounding errors break Python.
* **$f(x + \epsilon) - f(x - \epsilon)$**: The total change in the function's height across that tiny window.
* **$2\epsilon$**: The total width of the window along the x-axis.
* **Height / Width**: The classic rise-over-run slope formula.


### Why AI Cares (In plain terms)

* **Gradient Checking (Grad-Check):** When engineers write custom CUDA kernels or autograd operations for backpropagation, they use this exact centered numerical derivative to verify that their analytic calculus math has zero bugs.
* **Black-box Optimization:** When an AI system cannot backpropagate through an external environment or non-differentiable simulator, numerical perturbations guide the search.


### NumPy Implementation

In [ ]:
# 1. Define our target function (e.g., a simple loss curve)
def f(x):
    return x**2


# 2. Point where we want to measure the slope
x = 3.0

# 3. Tiny perturbation step (epsilon)
epsilon = 1e-5

# ==========================================
# Method A: Forward Difference
# ==========================================
# (f(x + eps) - f(x)) / eps
slope_forward = (f(x + epsilon) - f(x)) / epsilon

# ==========================================
# Method B: Centered Difference (Higher Accuracy)
# ==========================================
# (f(x + eps) - f(x - eps)) / (2 * eps)
slope_centered = (f(x + epsilon) - f(x - epsilon)) / (2.0 * epsilon)

# ==========================================
# Ground Truth: Exact Analytical Derivative d/dx (x^2) = 2x
# ==========================================
true_slope = 2.0 * x

print(f"Measuring slope of f(x) = x^2 at x = {x}\n")
print(f"True Analytic Slope       : {true_slope}")
print(
    f"Forward Difference Approx : {slope_forward:.6f} (Error: {abs(true_slope - slope_forward):.2e})"
)
print(
    f"Centered Difference Approx: {slope_centered:.6f} (Error: {abs(true_slope - slope_centered):.2e})"
)

Measuring slope of f(x) = x^2 at x = 3.0

True Analytic Slope       : 6.0
Forward Difference Approx : 6.000010 (Error: 1.00e-05)
Centered Difference Approx: 6.000000 (Error: 3.93e-11)


In [ ]:
# Styling
sns.set_theme(style="whitegrid")
palette = sns.color_palette("deep")


def f(x):
    return x**2


x0 = 3.0
y0 = f(x0)
true_slope = 2.0 * x0  # 6.0


@interact(
    eps=FloatSlider(
        min=0.2,
        max=1.5,
        step=0.1,
        value=0.8,
        description="Step (ε):",
        continuous_update=True,
    )
)
def plot_tight_finite_differences(eps=0.8):
    # Sample points on the curve
    x_fwd, y_fwd = x0 + eps, f(x0 + eps)
    x_back, y_back = x0 - eps, f(x0 - eps)

    slope_fwd = (y_fwd - y0) / eps
    slope_cent = (y_fwd - y_back) / (2.0 * eps)
    err_fwd = abs(true_slope - slope_fwd)
    err_cent = abs(true_slope - slope_cent)

    _fig, ax = plt.subplots(figsize=(8.5, 6.2))

    # Dynamic limits zoomed directly onto the active chord cuts
    x_min, x_max = x0 - 1.5 * eps, x0 + 1.5 * eps
    xs = np.linspace(x_min, x_max, 300)

    # 1. Target function curve
    ax.plot(xs, f(xs), color="black", linewidth=2.8, label=r"Target $f(x) = x^2$")

    # 2. True Tangent Line at x0
    ax.plot(
        xs,
        y0 + true_slope * (xs - x0),
        color="gray",
        linestyle=":",
        linewidth=2.5,
        label=rf"True Tangent ($m = {true_slope:.2f}$)",
    )

    # 3. Forward Difference Chord: [x0, x0 + eps]
    ax.plot(
        [x0, x_fwd],
        [y0, y_fwd],
        color=palette[3],
        linestyle="--",
        linewidth=2.8,
        label=rf"Forward Chord ($m = {slope_fwd:.2f}$, Err = {err_fwd:.2f})",
    )

    # 4. Centered Difference Chord: [x0 - eps, x0 + eps]
    ax.plot(
        [x_back, x_fwd],
        [y_back, y_fwd],
        color=palette[2],
        linestyle="-.",
        linewidth=2.8,
        label=rf"Centered Chord ($m = {slope_cent:.2f}$, Err = {err_cent:.2f})",
    )

    # 5. Rise-over-run triangle for Forward Difference
    ax.plot([x0, x_fwd], [y0, y0], color=palette[3], linestyle=":", linewidth=1.6)
    ax.plot([x_fwd, x_fwd], [y0, y_fwd], color=palette[3], linestyle=":", linewidth=1.6)
    ax.text(
        x0 + eps / 2,
        y0 - 0.08 * (y_fwd - y0),
        r"Run: $\Delta x = \epsilon$",
        color=palette[3],
        ha="center",
        va="top",
        fontsize=10,
        weight="bold",
    )
    ax.text(
        x_fwd + 0.03 * eps,
        (y0 + y_fwd) / 2,
        r"Rise: $\Delta y$",
        color=palette[3],
        ha="left",
        va="center",
        fontsize=10,
        weight="bold",
    )

    # 6. Evaluation Points
    ax.scatter(
        [x0], [y0], color="black", s=90, zorder=6, label=r"Evaluation Point $(x, f(x))$"
    )
    ax.scatter([x_fwd], [y_fwd], color=palette[3], s=90, zorder=6)
    ax.scatter([x_back], [y_back], color=palette[2], s=90, zorder=6)

    # Point labels
    ax.annotate(
        r"$(x, f(x))$",
        (x0, y0),
        textcoords="offset points",
        xytext=(-25, 12),
        weight="bold",
        fontsize=10,
    )
    ax.annotate(
        r"$(x+\epsilon, f(x+\epsilon))$",
        (x_fwd, y_fwd),
        textcoords="offset points",
        xytext=(-45, 12),
        weight="bold",
        color=palette[3],
        fontsize=10,
    )
    ax.annotate(
        r"$(x-\epsilon, f(x-\epsilon))$",
        (x_back, y_back),
        textcoords="offset points",
        xytext=(-45, -18),
        weight="bold",
        color=palette[2],
        fontsize=10,
    )

    # Enforce adaptive zoom bounds
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(max(0, y0 - 1.4 * (y0 - y_back)), y0 + 1.4 * (y_fwd - y0))

    ax.set_xlabel("x", fontsize=11)
    ax.set_ylabel("f(x)", fontsize=11)
    ax.set_title(
        rf"Finite Differences at $x = {x0:.1f}$ ($\epsilon = {eps:.2f}$)"
        + "\n"
        + r"Forward $\mathcal{O}(\epsilon)$ has upward bias | Centered $\mathcal{O}(\epsilon^2)$ cancels quadratic error",
        fontsize=12,
        weight="bold",
    )
    ax.legend(loc="upper left", frameon=True, fontsize=9.5)

    plt.tight_layout()
    plt.show()

interactive(children=(FloatSlider(value=0.8, description='Step (ε):', max=1.5, min=0.2), Output()), _dom_class…

### Concept 2: Partial Derivatives

### What is a Partial Derivative?

In Concept 1, the function only had **one** input knob to turn ($x$).

In machine learning, loss functions have **millions or billions of knobs** (weights $w_1, w_2, \dots, w_n$ and biases).

A **partial derivative** answers this specific question:

> *"If I tweak **only one specific knob** while freezing all other knobs completely still, how much does the output change?"*

* The word **"partial"** means you are only looking at **part** of the input space.
* If you have two inputs $(x, y)$, you can compute:
* The rate of change with respect to $x$ (pretending $y$ is just a constant number like $5$).
* The rate of change with respect to $y$ (pretending $x$ is just a constant number like $5$).


### The Formula

Instead of the standard letter $d$, mathematicians use the curly symbol $\partial$ (pronounced **"del"** or **"partial"**).

For a function of two variables $f(x, y)$:

**1. Partial derivative with respect to $x$:**

$$\frac{\partial f}{\partial x} \approx \frac{f(x + \epsilon, y) - f(x - \epsilon, y)}{2\epsilon}$$

**2. Partial derivative with respect to $y$:**

$$\frac{\partial f}{\partial y} \approx \frac{f(x, y + \epsilon) - f(x, y - \epsilon)}{2\epsilon}$$

#### How to read it out loud:

> *"The partial derivative of f with respect to x equals the limit of the change in f when only x is perturbed, divided by two epsilon."*

#### Breakdown of every piece:

* **$\partial f$**: The tiny change in the function's output.
* **$\partial x$**: The tiny nudge applied **only** along the $x$ direction.
* **$f(x + \epsilon, y)$**: Notice that $x$ moves forward by $\epsilon$, but $y$ remains frozen in place.
* **$f(x, y + \epsilon)$**: Notice that $y$ moves forward by $\epsilon$, but $x$ remains frozen in place.


### Why AI Cares (In plain terms)

Neural networks learn through **Backpropagation**.

When a model outputs an error on a batch of data, backpropagation calculates the **partial derivative of the loss with respect to every individual weight** ($\frac{\partial L}{\partial w_{ij}}$). This tells the optimizer which specific weights caused the error and how much each one needs to adjust.


Let's test this on a function with two inputs:


$$f(x, y) = x^2 + 3xy + y^3$$

Analytic rules:

* With respect to $x$ (treat $y$ as a constant): $\frac{\partial f}{\partial x} = 2x + 3y$
* With respect to $y$ (treat $x$ as a constant): $\frac{\partial f}{\partial y} = 3x + 3y^2$

Let's evaluate at point $(x = 2.0, y = 4.0)$:

* True $\frac{\partial f}{\partial x} = 2(2) + 3(4) = 4 + 12 = 16.0$
* True $\frac{\partial f}{\partial y} = 3(2) + 3(4^2) = 6 + 48 = 54.0$

### NumPy Implementation

In [ ]:
# 1. Define our multivariable function
def f(x, y):
    return (x**2) + (3.0 * x * y) + (y**3)


# 2. Coordinate where we evaluate the slopes
x = 2.0
y = 4.0

# 3. Small perturbation epsilon
epsilon = 1e-5

# ==========================================
# Partial derivative with respect to x (freeze y)
# ==========================================
df_dx = (f(x + epsilon, y) - f(x - epsilon, y)) / (2.0 * epsilon)

# ==========================================
# Partial derivative with respect to y (freeze x)
# ==========================================
df_dy = (f(x, y + epsilon) - f(x, y - epsilon)) / (2.0 * epsilon)

# Analytical solutions for validation
true_df_dx = 2.0 * x + 3.0 * y
true_df_dy = 3.0 * x + 3.0 * (y**2)

print(f"Evaluating f(x, y) = x^2 + 3xy + y^3 at (x={x}, y={y})\n")
print(f"Partial w.r.t x (Numerical): {df_dx:.6f} | True Analytic: {true_df_dx}")
print(f"Partial w.r.t y (Numerical): {df_dy:.6f} | True Analytic: {true_df_dy}")


Evaluating f(x, y) = x^2 + 3xy + y^3 at (x=2.0, y=4.0)

Partial w.r.t x (Numerical): 16.000000 | True Analytic: 16.0
Partial w.r.t y (Numerical): 54.000000 | True Analytic: 54.0


In [ ]:
# 1. Define function & point of interest
def f(x, y):
    return (x**2) + (3.0 * x * y) + (y**3)


x0, y0 = 2.0, 4.0
z0 = f(x0, y0)  # 92.0

# Analytical slopes (partial derivatives)
df_dx = 2.0 * x0 + 3.0 * y0  # 16.0
df_dy = 3.0 * x0 + 3.0 * (y0**2)  # 54.0

# 2. Smooth 3D surface mesh
x_grid = np.linspace(0.5, 3.5, 50)
y_grid = np.linspace(2.5, 5.5, 50)
X, Y = np.meshgrid(x_grid, y_grid)
Z = f(X, Y)

# 3. Tangent Plane around (x0, y0)
px = np.linspace(1.2, 2.8, 15)
py = np.linspace(3.2, 4.8, 15)
PX, PY = np.meshgrid(px, py)
PZ = z0 + df_dx * (PX - x0) + df_dy * (PY - y0)

# 4. Slices across each variable
# Slice along X (freeze y = 4.0)
sx = np.linspace(0.5, 3.5, 80)
sy = np.full_like(sx, y0)
sz = f(sx, sy)

# Slice along Y (freeze x = 2.0)
ty = np.linspace(2.5, 5.5, 80)
tx = np.full_like(ty, x0)
tz = f(tx, ty)

# 5. Build plot
fig = go.Figure()

# Base Surface
fig.add_trace(
    go.Surface(
        x=X,
        y=Y,
        z=Z,
        colorscale="Blues",
        opacity=0.6,
        showscale=False,
        name="Surface f(x, y)",
    )
)

# Tangent Plane Sheet
fig.add_trace(
    go.Surface(
        x=PX,
        y=PY,
        z=PZ,
        colorscale="Oranges",
        opacity=0.5,
        showscale=False,
        name="Tangent Plane",
    )
)

# 1D Slice Curves
fig.add_trace(
    go.Scatter3d(
        x=sx,
        y=sy,
        z=sz,
        mode="lines",
        line={"color": "#0055ff", "width": 7},
        name=f"Slice along X (∂f/∂x = {df_dx:.0f})",
    )
)

fig.add_trace(
    go.Scatter3d(
        x=tx,
        y=ty,
        z=tz,
        mode="lines",
        line={"color": "#ff3300", "width": 7},
        name=f"Slice along Y (∂f/∂y = {df_dy:.0f})",
    )
)

# Evaluation Point
fig.add_trace(
    go.Scatter3d(
        x=[x0],
        y=[y0],
        z=[z0],
        mode="markers+text",
        marker={"size": 7, "color": "black"},
        text=[f"P({x0}, {y0}, {z0:.0f})"],
        textposition="top center",
        name="Point (2, 4)",
    )
)

fig.update_layout(
    title=f"3D Partial Derivatives & Tangent Plane at (2, 4)<br>∂f/∂x = {df_dx:.1f} | ∂f/∂y = {df_dy:.1f}",
    scene={
        "xaxis_title": "X (Frozen for ∂f/∂y)",
        "yaxis_title": "Y (Frozen for ∂f/∂x)",
        "zaxis_title": "Z Height",
        "aspectratio": {"x": 1, "y": 1, "z": 0.8},  # Prevents vertical stretching
    },
    width=850,
    height=600,
    legend={"x": 0.02, "y": 0.98},
)

fig.show()

### Concept 3: The Gradient Vector ($\nabla f$)

### What is the Gradient Vector?

In Concept 2, you computed individual partial derivatives for each variable separately ($\frac{\partial f}{\partial x}$ and $\frac{\partial f}{\partial y}$).

The **Gradient Vector** simply bundles all those individual partial derivatives into a single list (a vector).

Imagine standing on a foggy hill:

* The slope along the East-West axis is one number.
* The slope along the North-South axis is another number.
* Put them together into an arrow, and that arrow points in the **direction of steepest ascent** (the fastest way uphill).

> **Two Golden Rules of the Gradient:**
> 1. **Direction:** The gradient arrow always points in the direction where the function increases the fastest.
> 2. **Length (Magnitude):** The length of the gradient vector tells you *how steep* that hill is.
> 
> 
> If you want to **minimize** a loss function (go downhill to the lowest point), you simply walk in the **exact opposite direction: $-\nabla f$** (this is Gradient Descent).

### The Formula

The symbol for the gradient is an upside-down triangle called **Nabla** ($\nabla$).

For a function $f(\mathbf{x})$ taking an $n$-dimensional input vector $\mathbf{x} = [x_1, x_2, \dots, x_n]^\top$:

$$\nabla f(\mathbf{x}) = \begin{bmatrix} \frac{\partial f}{\partial x_1} \\ \frac{\partial f}{\partial x_2} \\ \vdots \\ \frac{\partial f}{\partial x_n} \end{bmatrix}$$

#### How to read it out loud:

> *"The gradient of f with respect to x (nabla f of x) is the vector of all partial derivatives of f evaluated at point x."*

#### Breakdown of every piece:

* **$\nabla f$**: The gradient vector. Its length (number of elements) equals the number of parameters/weights in the function.
* **$\frac{\partial f}{\partial x_i}$**: The individual partial derivative with respect to the $i$-th parameter.
* **$\Vert{}\nabla f\Vert{}_2$**: The Euclidean length of the gradient vector, representing the steepness of the steepest slope.


### Why AI Cares (In plain terms)

* **Gradient Descent:** The single fundamental training rule for modern neural networks:

$$\mathbf{w}_{\text{new}} = \mathbf{w}_{\text{old}} - \eta \nabla f(\mathbf{w})$$



(where $\eta$ is the learning rate).
* **Vanishing / Exploding Gradients:** When $\Vert{}\nabla f\Vert{} \to 0$, the model stops learning because the gradient vector is too short. When $\Vert{}\nabla f\Vert{} \to \infty$, updates blow up and destroy model weights.

### NumPy Implementation

Here is an arbitrary $n$-dimensional numerical gradient calculator using centered differences that works for any function taking a vector of inputs:


In [ ]:
import numpy as np


# 1. Define a multivariable scalar loss function: f(w) = w_0^2 + 3*w_0*w_1 + w_1^3
def loss_fn(w):
    return (w[0] ** 2) + (3.0 * w[0] * w[1]) + (w[1] ** 3)


# 2. General-purpose numerical gradient function
def compute_gradient(f, w, epsilon=1e-5):
    """
    Computes the gradient vector of f at point w using centered differences.
    Works for any arbitrary dimension of w.
    """
    grad = np.zeros_like(w, dtype=float)

    # Loop over each parameter index to nudge it individually
    for i in range(len(w)):
        w_plus = w.copy()
        w_minus = w.copy()

        # Nudge ONLY parameter i forward and backward
        w_plus[i] += epsilon
        w_minus[i] -= epsilon

        # Centered difference formula for partial derivative i
        grad[i] = (f(w_plus) - f(w_minus)) / (2.0 * epsilon)

    return grad


# 3. Test at point w = [2.0, 4.0]
w = np.array([2.0, 4.0])
grad = compute_gradient(loss_fn, w)

# From Concept 2: true df/dw0 = 16.0, true df/dw1 = 54.0
print(f"Current weight vector w:        {w}")
print(f"Computed Gradient Vector (∇f):  {grad}")
print(f"Direction of steepest descent:  {-grad}")
print(f"Gradient Magnitude (steepness): {np.linalg.norm(grad):.4f}")


Current weight vector w:        [2. 4.]
Computed Gradient Vector (∇f):  [16. 54.]
Direction of steepest descent:  [-16. -54.]
Gradient Magnitude (steepness): 56.3205


In [15]:
# 1. Target function & gradients
def loss_fn(w0, w1):
    return (w0**2) + (3.0 * w0 * w1) + (w1**3)


w0_pt, w1_pt = 2.0, 4.0
z_pt = loss_fn(w0_pt, w1_pt)  # 92.0

# Analytical gradient components: [16.0, 54.0]
g0 = 2.0 * w0_pt + 3.0 * w1_pt
g1 = 3.0 * w0_pt + 3.0 * (w1_pt**2)
grad_2d = np.array([g0, g1])
norm_grad = np.linalg.norm(grad_2d)
dir_2d = grad_2d / norm_grad  # Normalized direction on the w0-w1 plane

# 2. Smooth 3D surface grid
w0_vals = np.linspace(0.5, 3.5, 45)
w1_vals = np.linspace(2.5, 5.5, 45)
W0, W1 = np.meshgrid(w0_vals, w1_vals)
Z = loss_fn(W0, W1)
z_floor = float(np.min(Z))  # Level of bottom contour plane

# 3. Tangent directional step on the 3D surface
# Moving by step 'delta' in (w0, w1) changes height by: dz = g0*dw0 + g1*dw1
delta = 0.5
dz = g0 * (dir_2d[0] * delta) + g1 * (dir_2d[1] * delta)

# Endpoints for 3D Surface Tangent Vectors
ascent_end_3d = np.array(
    [w0_pt + dir_2d[0] * delta, w1_pt + dir_2d[1] * delta, z_pt + dz]
)
descent_end_3d = np.array(
    [w0_pt - dir_2d[0] * delta, w1_pt - dir_2d[1] * delta, z_pt - dz]
)

# Endpoints for 2D Floor Arrows (projected on contour base)
ascent_end_floor = np.array(
    [w0_pt + dir_2d[0] * delta, w1_pt + dir_2d[1] * delta, z_floor]
)
descent_end_floor = np.array(
    [w0_pt - dir_2d[0] * delta, w1_pt - dir_2d[1] * delta, z_floor]
)


# Helper to build vector lines + cones
def make_vector(start, end, color, name, show_in_legend=True):
    line = go.Scatter3d(
        x=[start[0], end[0]],
        y=[start[1], end[1]],
        z=[start[2], end[2]],
        mode="lines",
        line={"color": color, "width": 6},
        name=name,
        showlegend=show_in_legend,
    )
    direction = end - start
    cone = go.Cone(
        x=[end[0]],
        y=[end[1]],
        z=[end[2]],
        u=[direction[0]],
        v=[direction[1]],
        w=[direction[2]],
        sizemode="absolute",
        sizeref=0.22,
        colorscale=[[0, color], [1, color]],
        showscale=False,
        hoverinfo="skip",
    )
    return line, cone


# 3D surface arrows (hugging the slope)
surf_asc_line, surf_asc_cone = make_vector(
    np.array([w0_pt, w1_pt, z_pt]), ascent_end_3d, "#d62728", "Steepest Ascent (+∇f)"
)
surf_desc_line, surf_desc_cone = make_vector(
    np.array([w0_pt, w1_pt, z_pt]), descent_end_3d, "#2ca02c", "Steepest Descent (-∇f)"
)

# 2D base floor arrows (pointing across contour lines)
floor_asc_line, floor_asc_cone = make_vector(
    np.array([w0_pt, w1_pt, z_floor]),
    ascent_end_floor,
    "#d62728",
    "Floor Direction (+∇f)",
    False,
)
floor_desc_line, floor_desc_cone = make_vector(
    np.array([w0_pt, w1_pt, z_floor]),
    descent_end_floor,
    "#2ca02c",
    "Floor Direction (-∇f)",
    False,
)

# Drop line connecting surface point to floor
drop_line = go.Scatter3d(
    x=[w0_pt, w0_pt],
    y=[w1_pt, w1_pt],
    z=[z_floor, z_pt],
    mode="lines",
    line={"color": "black", "dash": "dash", "width": 2},
    name="Projection Drop",
)

# 4. Build Figure
fig = go.Figure()

# Surface with bottom projected contours
fig.add_trace(
    go.Surface(
        x=W0,
        y=W1,
        z=Z,
        colorscale="Blues",
        opacity=0.6,
        showscale=False,
        contours={
            "z": {
                "show": True, "project_z": True, "usecolormap": True, "highlightcolor": "#2ca02c"
            }
        },
        name="Surface f(w)",
    )
)

# Add traces
fig.add_trace(drop_line)
fig.add_trace(surf_asc_line)
fig.add_trace(surf_asc_cone)
fig.add_trace(surf_desc_line)
fig.add_trace(surf_desc_cone)
fig.add_trace(floor_asc_line)
fig.add_trace(floor_asc_cone)
fig.add_trace(floor_desc_line)
fig.add_trace(floor_desc_cone)

# Evaluation points on surface and floor
fig.add_trace(
    go.Scatter3d(
        x=[w0_pt, w0_pt],
        y=[w1_pt, w1_pt],
        z=[z_pt, z_floor],
        mode="markers+text",
        marker={"size": [7, 5], "color": ["black", "black"]},
        text=[f"w(2, 4, {z_pt:.0f})", "w_base(2, 4)"],
        textposition="top left",
        name="Eval Point w",
    )
)

fig.update_layout(
    title=f"Gradient Vectors on Surface & Floor<br>∇f = [{g0:.1f}, {g1:.1f}] | Steepest Descent = [{-g0:.1f}, {-g1:.1f}]",
    scene={
        "xaxis_title": "w₀",
        "yaxis_title": "w₁",
        "zaxis_title": "Loss f(w)",
        "aspectratio": {"x": 1, "y": 1, "z": 0.75},
        "camera": {"eye": {"x": -1.6, "y": -1.5, "z": 1.2}},  # Better starting perspective
    },
    width=900,
    height=650,
    legend={"x": 0.02, "y": 0.95},
)

fig.show()

### Concept 4: Directional Derivatives

### What is a Directional Derivative?

In Concept 2, you measured slopes strictly along the predefined axes: pure horizontal ($\frac{\partial f}{\partial x}$) or pure vertical ($\frac{\partial f}{\partial y}$).

In Concept 3, the gradient gave you the **single direction of steepest slope**.

**The Directional Derivative answers a flexible question:**

> *"What if I don't want to walk along the x-axis, the y-axis, or even the steepest path? What if I choose to walk along **any custom arbitrary direction** $\mathbf{v}$? What slope will I feel under my feet?"*

* It measures the instantaneous rate of change of $f$ as you move through point $\mathbf{x}$ in the direction of a unit vector $\mathbf{v}$.
* If you point $\mathbf{v}$ directly uphill (along $\nabla f$), you get the maximum possible positive slope.
* If you point $\mathbf{v}$ perpendicular to the hill (orthogonal to $\nabla f$), your slope is **0** (walking along a flat contour line).


### The Formula

There is a remarkably elegant shortcut: you do **not** need to do complex new calculus to find this slope. The directional derivative along any unit direction $\mathbf{v}$ is simply the **Dot Product** (Concept 1) between the **Gradient Vector** (Concept 3) and that direction vector:

$$D_{\mathbf{v}} f(\mathbf{x}) = \nabla f(\mathbf{x}) \cdot \mathbf{v}$$

#### How to read it out loud:

> *"The directional derivative of f in the direction of unit vector v equals the dot product of the gradient nabla f and vector v."*

#### Breakdown of every piece:

* **$D_{\mathbf{v}} f(\mathbf{x})$**: The slope (a single scalar number) you experience when moving along direction $\mathbf{v}$.
* **$\nabla f(\mathbf{x})$**: The Gradient Vector at point $\mathbf{x}$.
* **$\mathbf{v}$**: A **unit vector** pointing in the desired travel direction ($\Vert{}\mathbf{v}\Vert{}_2 = 1$). It must be normalized so that its length doesn't artificially inflate the slope.
* **$\cdot$**: The standard vector dot product ($\sum \nabla f_i \cdot v_i$).

Because $\nabla f \cdot \mathbf{v} = \Vert{}\nabla f\Vert{} \Vert{}\mathbf{v}\Vert{} \cos(\theta) = \Vert{}\nabla f\Vert{} \cos(\theta)$:

* When $\mathbf{v}$ aligns with $\nabla f$ ($\theta = 0^\circ$), $\cos(0) = 1$, giving the maximum uphill slope $\Vert{}\nabla f\Vert{}$.
* When $\mathbf{v}$ points opposite ($180^\circ$), $\cos(180) = -1$, giving the steepest descent $-\Vert{}\nabla f\Vert{}$.


### Why AI Cares (In plain terms)

* **Line Search Optimizers:** Advanced optimization algorithms (such as L-BFGS, Conjugate Gradient, and Wolfe-condition line searches) do not take fixed-size gradient steps. They pick a search direction $\mathbf{v}$ and evaluate directional derivatives along that trajectory to calculate how far to step before stopping.
* **Loss Landscape Analysis:** When researchers visualize deep neural network loss surfaces (measuring "flat minima" vs "sharp minima"), they evaluate directional derivatives along random 1D or 2D slices of the multi-billion-parameter weight space.


### NumPy Implementation


In [ ]:
import numpy as np


# 1. Define our scalar function: f(w) = w_0^2 + 3*w_0*w_1 + w_1^3
def loss_fn(w):
    return (w[0] ** 2) + (3.0 * w[0] * w[1]) + (w[1] ** 3)


# 2. Re-use our numerical gradient function from Concept 3
def compute_gradient(f, w, epsilon=1e-5):
    grad = np.zeros_like(w, dtype=float)
    for i in range(len(w)):
        w_plus = w.copy()
        w_minus = w.copy()
        w_plus[i] += epsilon
        w_minus[i] -= epsilon
        grad[i] = (f(w_plus) - f(w_minus)) / (2.0 * epsilon)
    return grad


# Point where we stand
w = np.array([2.0, 4.0])
grad = compute_gradient(loss_fn, w)  # Gradient: [16.0, 54.0]

# ==========================================
# Choose a custom arbitrary direction: e.g., move along [1.0, 1.0]
# ==========================================
raw_direction = np.array([1.0, 1.0])

# CRITICAL STEP: Direction must be a unit vector (norm = 1)
unit_v = raw_direction / np.linalg.norm(raw_direction)

# ==========================================
# Method A: Dot Product Formula: D_v(f) = ∇f · v
# ==========================================
directional_derivative_formula = np.dot(grad, unit_v)

# ==========================================
# Method B: Direct Numerical Perturbation along v
# (f(w + eps * v) - f(w - eps * v)) / (2 * eps)
# ==========================================
eps = 1e-5
directional_derivative_numerical = (
    loss_fn(w + eps * unit_v) - loss_fn(w - eps * unit_v)
) / (2.0 * eps)

# ==========================================
# Compare with the absolute maximum uphill slope
# ==========================================
max_slope = np.linalg.norm(grad)

print(f"Gradient at point w:               {grad}")
print(f"Custom Unit Direction (unit_v):    {unit_v}")
print(f"\nDirectional Derivative (Formula):  {directional_derivative_formula:.5f}")
print(f"Directional Derivative (Numerical):{directional_derivative_numerical:.5f}")
print(f"Maximum Possible Slope (along ∇f): {max_slope:.5f}")


Gradient at point w:               [16. 54.]
Custom Unit Direction (unit_v):    [0.70710678 0.70710678]

Directional Derivative (Formula):  49.49747
Directional Derivative (Numerical):49.49747
Maximum Possible Slope (along ∇f): 56.32051


In [ ]:
def loss_fn(w0, w1):
    return (w0**2) + (3.0 * w0 * w1) + (w1**3)


w0_pt, w1_pt = 2.0, 4.0
z_pt = loss_fn(w0_pt, w1_pt)  # 92.0

# Gradients
g0 = 2.0 * w0_pt + 3.0 * w1_pt
g1 = 3.0 * w0_pt + 3.0 * (w1_pt**2)
grad = np.array([g0, g1])
max_slope = np.linalg.norm(grad)

# Custom unit direction v = [1, 1] normalized
unit_v = np.array([1.0, 1.0]) / np.sqrt(2.0)
slope_v = np.dot(grad, unit_v)

# Surface grid
w0_vals = np.linspace(0.8, 3.2, 40)
w1_vals = np.linspace(2.8, 5.2, 40)
W0, W1 = np.meshgrid(w0_vals, w1_vals)
Z = loss_fn(W0, W1)
z_min = float(np.min(Z))

# 1. Path walked along direction v across the surface
t_v = np.linspace(-0.8, 0.8, 60)
path_v_w0 = w0_pt + t_v * unit_v[0]
path_v_w1 = w1_pt + t_v * unit_v[1]
path_v_z = loss_fn(path_v_w0, path_v_w1)

# 2. Path walked along true gradient (steepest path)
dir_grad = grad / max_slope
t_g = np.linspace(-0.8, 0.8, 60)
path_g_w0 = w0_pt + t_g * dir_grad[0]
path_g_w1 = w1_pt + t_g * dir_grad[1]
path_g_z = loss_fn(path_g_w0, path_g_w1)

fig = go.Figure()

# Background 3D Surface
fig.add_trace(
    go.Surface(
        x=W0,
        y=W1,
        z=Z,
        colorscale="Blues",
        opacity=0.5,
        showscale=False,
        name="Loss Surface",
    )
)

# Purple Path: Walking along chosen direction v
fig.add_trace(
    go.Scatter3d(
        x=path_v_w0,
        y=path_v_w1,
        z=path_v_z,
        mode="lines",
        line={"color": "#9467bd", "width": 8},
        name=f"Path along v (Slope = {slope_v:.1f})",
    )
)

# Red Path: Walking along steepest gradient direction
fig.add_trace(
    go.Scatter3d(
        x=path_g_w0,
        y=path_g_w1,
        z=path_g_z,
        mode="lines",
        line={"color": "#d62728", "width": 5, "dash": "dash"},
        name=f"Steepest Path ∇f (Max Slope = {max_slope:.1f})",
    )
)

# Current Point
fig.add_trace(
    go.Scatter3d(
        x=[w0_pt],
        y=[w1_pt],
        z=[z_pt],
        mode="markers+text",
        marker={"size": 8, "color": "black"},
        text=["w = (2, 4)"],
        textposition="top center",
        name="Current Point",
    )
)

fig.update_layout(
    title=f"Directional Derivative: Climbing along v vs ∇f<br>"
    f"Slope along v = {slope_v:.2f}  |  Maximum possible slope along ∇f = {max_slope:.2f}",
    scene={
        "xaxis_title": "w₀",
        "yaxis_title": "w₁",
        "zaxis_title": "Loss f(w)",
        "aspectratio": {"x": 1, "y": 1, "z": 0.75},
        "camera": {"eye": {"x": -1.5, "y": -1.5, "z": 1.0}},
    },
    width=850,
    height=600,
    legend={"x": 0.02, "y": 0.95},
)

fig.show()

### Concept 5: The Jacobian Matrix

### What is the Jacobian Matrix? (Simple English)

So far in Calculus:

* **Derivative / Gradient:** Took multiple input knobs ($\mathbf{x}$), but produced **only 1 output number** (like a single Loss score).

In the real world, neural network layers do not output a single number. A hidden layer takes an input vector of size $n$ and outputs an **entire vector** of size $m$ (e.g., passing a token through an activation function or a linear projection).

**The Jacobian Matrix answers this question:**

> *"If I have a function that takes a vector of inputs and produces a vector of outputs, how does **every single output element** change when I tweak **every single input knob**?"*

* It is a complete grid (matrix) of all possible first-order partial derivatives.
* If you have $m$ outputs and $n$ inputs, the Jacobian is an $(m \times n)$ matrix.
* Each row is simply the **gradient of one specific output element**.


### The Formula

For a vector-valued function $\mathbf{f}(\mathbf{x})$ that maps an $n$-dimensional input $\mathbf{x} = [x_1, x_2, \dots, x_n]^\top$ to an $m$-dimensional output $\mathbf{f} = [f_1, f_2, \dots, f_m]^\top$:

$$J = \frac{\partial \mathbf{f}}{\partial \mathbf{x}} = \begin{bmatrix} \frac{\partial f_1}{\partial x_1} & \frac{\partial f_1}{\partial x_2} & \cdots & \frac{\partial f_1}{\partial x_n} \\ \frac{\partial f_2}{\partial x_1} & \frac{\partial f_2}{\partial x_2} & \cdots & \frac{\partial f_2}{\partial x_n} \\ \vdots & \vdots & \ddots & \vdots \\ \frac{\partial f_m}{\partial x_1} & \frac{\partial f_m}{\partial x_2} & \cdots & \frac{\partial f_m}{\partial x_n} \end{bmatrix}$$

#### How to read it out loud:

> *"The Jacobian matrix J is the m-by-n matrix where the entry in row i and column j is the partial derivative of output f_i with respect to input x_j."*

#### Breakdown of every piece:

* **Row $i$**: How output component $f_i$ responds to changes across all inputs ($x_1, x_2, \dots, x_n$). This row is literally $(\nabla f_i)^\top$.
* **Column $j$**: How changing a single input $x_j$ cascades across all outputs ($f_1, f_2, \dots, f_m$).
* **Entry $J_{i, j} = \frac{\partial f_i}{\partial x_j}$**: The rate of change of output $i$ specifically due to input $j$.


### Why AI Cares (In plain terms)

* **Multivariate Chain Rule:** Backpropagation is literally chained matrix multiplications of Jacobians. To push gradients backward through a layer $\mathbf{y} = \mathbf{f}(\mathbf{x})$, PyTorch computes:

$$\frac{\partial L}{\partial \mathbf{x}} = \left(\frac{\partial \mathbf{y}}{\partial \mathbf{x}}\right)^\top \frac{\partial L}{\partial \mathbf{y}} = J^\top \nabla_{\mathbf{y}} L$$


* **Vector-Jacobian Products (VJPs):** Frameworks like PyTorch Autograd and JAX never store the giant full Jacobian matrix in memory. Instead, they directly calculate the product of an upstream gradient vector with the Jacobian ($v^\top J$), saving immense amounts of VRAM.
* **Softmax Derivative:** Deriving how every softmax probability changes with respect to every input logit requires calculating an $(n \times n)$ Jacobian matrix.


### NumPy Implementation

Let's test this on a function that takes a 3D input vector $\mathbf{x} = [x_0, x_1, x_2]$ and outputs a 2D vector $\mathbf{f}(\mathbf{x}) = [f_0, f_1]$:

$$f_0(\mathbf{x}) = x_0^2 + 2x_1$$

$$f_1(\mathbf{x}) = x_0 x_1 + \sin(x_2)$$

Expected analytic Jacobian at any point:


$$J = \begin{bmatrix} \frac{\partial f_0}{\partial x_0} & \frac{\partial f_0}{\partial x_1} & \frac{\partial f_0}{\partial x_2} \\ \frac{\partial f_1}{\partial x_0} & \frac{\partial f_1}{\partial x_1} & \frac{\partial f_1}{\partial x_2} \end{bmatrix} = \begin{bmatrix} 2x_0 & 2 & 0 \\ x_1 & x_0 & \cos(x_2) \end{bmatrix}$$


In [16]:
# 1. Define a vector-in, vector-out function: R^3 -> R^2
def f(x):
    f0 = (x[0] ** 2) + (2.0 * x[1])
    f1 = (x[0] * x[1]) + np.sin(x[2])
    return np.array([f0, f1])


# 2. General-purpose numerical Jacobian calculator
def compute_jacobian(fn, x, epsilon=1e-5):
    """
    Computes the (m x n) Jacobian matrix of fn at point x using centered differences.
    m = output dimension
    n = input dimension
    """
    # Evaluate at current point to get output dimension m
    y0 = fn(x)
    m = len(y0)
    n = len(x)

    # Initialize Jacobian matrix of shape (m, n)
    J = np.zeros((m, n), dtype=float)

    # Nudge each input coordinate j one by one
    for j in range(n):
        x_plus = x.copy()
        x_minus = x.copy()

        x_plus[j] += epsilon
        x_minus[j] -= epsilon

        # Output changes across all m outputs due to input j
        diff = (fn(x_plus) - fn(x_minus)) / (2.0 * epsilon)

        # Store in column j of the Jacobian
        J[:, j] = diff

    return J


# 3. Test at point x = [1.5, 3.0, 0.0]
x = np.array([1.5, 3.0, 0.0])
J_numerical = compute_jacobian(f, x)

# Analytical Jacobian for exact comparison:
# Row 0: [2*x0, 2, 0]           -> [2*1.5, 2, 0]      -> [3.0, 2.0, 0.0]
# Row 1: [x1, x0, cos(x2)]      -> [3.0, 1.5, cos(0)] -> [3.0, 1.5, 1.0]
J_analytic = np.array([[2.0 * x[0], 2.0, 0.0], [x[1], x[0], np.cos(x[2])]])

print(f"Input vector x: {x}")
print(f"Function output f(x): {f(x)}\n")
print(f"Numerical Jacobian (Shape: {J_numerical.shape}):\n{np.round(J_numerical, 4)}\n")
print(f"Analytic Jacobian:\n{J_analytic}\n")
print(
    "Are they equal within numerical precision?", np.allclose(J_numerical, J_analytic)
)

Input vector x: [1.5 3.  0. ]
Function output f(x): [8.25 4.5 ]

Numerical Jacobian (Shape: (2, 3)):
[[3.  2.  0. ]
 [3.  1.5 1. ]]

Analytic Jacobian:
[[3.  2.  0. ]
 [3.  1.5 1. ]]

Are they equal within numerical precision? True


In [ ]:
sns.set_theme(style="whitegrid")
palette = sns.color_palette("deep")


def f(x):
    f0 = (x[0] ** 2) + (2.0 * x[1])
    f1 = (x[0] * x[1]) + np.sin(x[2])
    return np.array([f0, f1])


x0_base = np.array([1.5, 3.0, 0.0])
y0_base = f(x0_base)

# Exact Jacobian matrix at x0_base: shape (2, 3)
J = np.array(
    [[2.0 * x0_base[0], 2.0, 0.0], [x0_base[1], x0_base[0], np.cos(x0_base[2])]]
)


@interact(
    dx0=FloatSlider(min=-0.5, max=0.5, step=0.05, value=0.1, description="Δx₀:"),
    dx1=FloatSlider(min=-0.5, max=0.5, step=0.05, value=0.1, description="Δx₁:"),
    dx2=FloatSlider(min=-0.5, max=0.5, step=0.05, value=0.1, description="Δx₂:"),
)
def visualize_jacobian_action(dx0=0.1, dx1=0.1, dx2=0.1):
    delta_x = np.array([dx0, dx1, dx2])

    # 1. Linear approximation via Jacobian: Δy ≈ J @ Δx
    delta_y_linear = J @ delta_x

    # 2. True nonlinear change: Δy = f(x0 + Δx) - f(x0)
    delta_y_true = f(x0_base + delta_x) - y0_base

    fig, axes = plt.subplots(
        1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.1, 1]}
    )

    # --- Left: Jacobian Matrix Heatmap ---
    sns.heatmap(
        J,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        cbar=False,
        linewidths=1.2,
        linecolor="gray",
        square=True,
        ax=axes[0],
        annot_kws={"size": 12, "weight": "bold"},
    )
    axes[0].set_title(
        r"Jacobian Matrix $J = \left[\frac{\partial f_i}{\partial x_j}\right]$ (Shape: 2×3)",
        fontsize=11,
        weight="bold",
    )
    axes[0].set_xticklabels([r"$x_0$", r"$x_1$", r"$x_2$"], fontsize=11, weight="bold")
    axes[0].set_yticklabels(
        [r"$f_0$", r"$f_1$"], fontsize=11, weight="bold", rotation=0
    )

    # --- Right: Linear Approximation vs True Response ---
    indices = np.array([0, 1])
    bar_width = 0.35

    b1 = axes[1].bar(
        indices - bar_width / 2,
        delta_y_linear,
        bar_width,
        color=palette[0],
        label=r"Jacobian Prediction ($J \Delta \mathbf{x}$)",
    )
    b2 = axes[1].bar(
        indices + bar_width / 2,
        delta_y_true,
        bar_width,
        color=palette[2],
        label=r"Actual True Change ($\Delta \mathbf{y}$)",
    )

    # Value tags
    for b in b1:
        h = b.get_height()
        va = "bottom" if h >= 0 else "top"
        axes[1].text(
            b.get_x() + b.get_width() / 2,
            h,
            f"{h:+.3f}",
            ha="center",
            va=va,
            fontsize=9,
            weight="bold",
        )

    for b in b2:
        h = b.get_height()
        va = "bottom" if h >= 0 else "top"
        axes[1].text(
            b.get_x() + b.get_width() / 2,
            h,
            f"{h:+.3f}",
            ha="center",
            va=va,
            fontsize=9,
            weight="bold",
        )

    axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.6)
    axes[1].set_xticks(indices)
    axes[1].set_xticklabels(
        [r"$\Delta f_0$", r"$\Delta f_1$"], fontsize=11, weight="bold"
    )
    axes[1].set_ylabel(r"Change in Outputs ($\Delta y$)")
    axes[1].set_title(
        r"Local Linear Mapping: $\Delta \mathbf{y} \approx J \Delta \mathbf{x}$",
        fontsize=11,
        weight="bold",
    )
    axes[1].legend(loc="upper left", frameon=True, fontsize=9)

    fig.suptitle(
        rf"Input Shift $\Delta \mathbf{{x}} = [{dx0:+.2f}, {dx1:+.2f}, {dx2:+.2f}]^T$ at $\mathbf{{x}} = [1.5, 3.0, 0.0]^T$",
        fontsize=12,
        weight="bold",
        y=1.03,
    )

    plt.tight_layout()
    plt.show()

interactive(children=(FloatSlider(value=0.1, description='Δx₀:', max=0.5, min=-0.5, step=0.05), FloatSlider(va…

### Concept 6: The Hessian Matrix


### What is the Hessian Matrix?

In Concept 3, the **Gradient** told you the **slope** (first derivative): *"Which way is downhill, and how steep is it?"*

However, knowing the slope isn't enough when driving a car. You also need to know the **curvature of the road**:

* Is the slope staying constant (a flat ramp)?
* Is it curving upward sharply (a tight valley)?
* Is it dropping off into an abyss (a cliff edge)?

**The Hessian Matrix answers this question:**

> *"What is the **curvature** (second derivative) of the loss landscape in every possible combination of directions?"*

* While the gradient is a **vector** of first derivatives, the Hessian is a **square matrix** ($n \times n$) of all second-order partial derivatives.
* It measures how the gradient itself changes as you move through space.

> **Why Curvature Matters for Stepping:**
> * If curvature is **steep** (high second derivative), a large learning rate will cause you to overshoot and explode.
> * If curvature is **flat** (low second derivative), you can safely take giant steps without overshooting.
> 
> 

### The Formula

For a scalar loss function $f(\mathbf{x})$ taking an $n$-dimensional vector $\mathbf{x} = [x_1, x_2, \dots, x_n]^\top$:

$$H = \nabla^2 f(\mathbf{x}) = \begin{bmatrix} \frac{\partial^2 f}{\partial x_1^2} & \frac{\partial^2 f}{\partial x_1 \partial x_2} & \cdots & \frac{\partial^2 f}{\partial x_1 \partial x_n} \\ \frac{\partial^2 f}{\partial x_2 \partial x_1} & \frac{\partial^2 f}{\partial x_2^2} & \cdots & \frac{\partial^2 f}{\partial x_2 \partial x_n} \\ \vdots & \vdots & \ddots & \vdots \\ \frac{\partial^2 f}{\partial x_n \partial x_1} & \frac{\partial^2 f}{\partial x_n \partial x_2} & \cdots & \frac{\partial^2 f}{\partial x_n^2} \end{bmatrix}$$

#### How to read it out loud:

> *"The Hessian matrix H (nabla squared f) is the n-by-n square matrix where entry (i, j) is the second partial derivative of f with respect to x_i and x_j."*

#### Breakdown of every piece:

* **Diagonal entries ($\frac{\partial^2 f}{\partial x_i^2}$):** Curvature along each coordinate axis alone.
* **Off-diagonal entries ($\frac{\partial^2 f}{\partial x_i \partial x_j}$):** Cross-curvature—how turning knob $j$ changes the slope along knob $i$.
* **Symmetry:** By Schwarz's theorem, for smooth functions, the order of differentiation doesn't matter: $\frac{\partial^2 f}{\partial x_i \partial x_j} = \frac{\partial^2 f}{\partial x_j \partial x_i}$. Therefore, **the Hessian is always a symmetric matrix ($H = H^\top$)**.
* **Connection to Jacobians:** The Hessian of a scalar function $f$ is literally the **Jacobian of its gradient vector**: $H = J(\nabla f)$.


### Why AI Cares (In plain terms)

* **Detecting Saddle Points vs. Minima:** When the gradient is zero ($\nabla f = 0$), you are at a stationary point. The eigenvalues of the Hessian tell you what kind:
* All eigenvalues $> 0$ (Positive Definite): **Local Minimum** (bowl curving upward everywhere).
* All eigenvalues $< 0$ (Negative Definite): **Local Maximum** (dome curving downward everywhere).
* Mixed positive and negative eigenvalues: **Saddle Point** (curves up in one direction, down in another). Deep networks are packed with saddle points.


* **Second-Order Optimizers (Newton-Raphson):** Instead of guessing a learning rate, Newton's method uses the inverted Hessian to jump directly to the minimum:

$$\mathbf{w}_{\text{new}} = \mathbf{w}_{\text{old}} - H^{-1} \nabla f(\mathbf{w})$$


* **AdaHessian / Sophia:** Modern LLM optimizers approximate the diagonal of the Hessian to stabilize pretraining on massive parameter scales.


### NumPy Implementation

Let's test this on the multivariable function:


$$f(x_0, x_1) = x_0^3 + x_0 x_1^2 + 5x_1^2$$

Analytic second partial derivatives:

1. First derivatives:
* $\frac{\partial f}{\partial x_0} = 3x_0^2 + x_1^2$
* $\frac{\partial f}{\partial x_1} = 2x_0 x_1 + 10x_1$


2. Second derivatives (The Hessian):
* $H_{0,0} = \frac{\partial^2 f}{\partial x_0^2} = 6x_0$
* $H_{0,1} = \frac{\partial^2 f}{\partial x_0 \partial x_1} = 2x_1$
* $H_{1,0} = \frac{\partial^2 f}{\partial x_1 \partial x_0} = 2x_1$
* $H_{1,1} = \frac{\partial^2 f}{\partial x_1^2} = 2x_0 + 10$



In [ ]:
# 1. Define a scalar multivariable function
def f(x):
    return (x[0] ** 3) + (x[0] * (x[1] ** 2)) + (5.0 * (x[1] ** 2))


# 2. Re-use numerical gradient to compute Hessian as Jacobian of the gradient
def compute_gradient(fn, x, epsilon=1e-5):
    grad = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        x_plus = x.copy()
        x_minus = x.copy()
        x_plus[i] += epsilon
        x_minus[i] -= epsilon
        grad[i] = (fn(x_plus) - fn(x_minus)) / (2.0 * epsilon)
    return grad


def compute_hessian(fn, x, epsilon=1e-4):
    """
    Computes the (n x n) Hessian matrix of scalar fn at point x.
    Uses centered difference of the gradient: H = J(∇f).
    """
    n = len(x)
    H = np.zeros((n, n), dtype=float)

    # Nudge each parameter j and see how the entire gradient vector responds
    for j in range(n):
        x_plus = x.copy()
        x_minus = x.copy()
        x_plus[j] += epsilon
        x_minus[j] -= epsilon

        # Gradients at nudged points
        grad_plus = compute_gradient(fn, x_plus)
        grad_minus = compute_gradient(fn, x_minus)

        # Second derivative column: change in gradient / (2 * eps)
        H[:, j] = (grad_plus - grad_minus) / (2.0 * epsilon)

    return H


# 3. Test at point x = [2.0, 3.0]
x = np.array([2.0, 3.0])
H_numerical = compute_hessian(f, x)

# Analytic Hessian evaluated at x = [2.0, 3.0]:
# H00 = 6*(2.0) = 12.0
# H01 = 2*(3.0) = 6.0
# H10 = 2*(3.0) = 6.0
# H11 = 2*(2.0) + 10 = 14.0
H_analytic = np.array([[6.0 * x[0], 2.0 * x[1]], [2.0 * x[1], 2.0 * x[0] + 10.0]])

# 4. Check eigenvalues of Hessian to inspect curvature type
eigenvalues = np.linalg.eigvals(H_numerical)

print(f"Point x: {x}\n")
print(f"Numerical Hessian Matrix:\n{np.round(H_numerical, 4)}\n")
print(f"Analytic Hessian Matrix:\n{H_analytic}\n")
print(f"Hessian Eigenvalues: {eigenvalues}")

if np.all(eigenvalues > 0):
    print("Curvature: Positive Definite (Strict local valley / upward bowl)")
elif np.all(eigenvalues < 0):
    print("Curvature: Negative Definite (Peak / downward dome)")
else:
    print("Curvature: Indefinite (Saddle point)")

Point x: [2. 3.]

Numerical Hessian Matrix:
[[12.  6.]
 [ 6. 14.]]

Analytic Hessian Matrix:
[[12.  6.]
 [ 6. 14.]]

Hessian Eigenvalues: [ 6.91723449+0.j 19.08276056+0.j]
Curvature: Positive Definite (Strict local valley / upward bowl)


In [ ]:
# 1. Target function and point
def f(x0, x1):
    return (x0**3) + (x0 * (x1**2)) + (5.0 * (x1**2))


x0_pt, x1_pt = 2.0, 3.0
z_pt = f(x0_pt, x1_pt)  # 2^3 + 2*(3^2) + 5*(3^2) = 8 + 18 + 45 = 71.0

# First derivatives (Gradient)
g0 = 3.0 * (x0_pt**2) + (x1_pt**2)  # 3(4) + 9 = 21.0
g1 = 2.0 * x0_pt * x1_pt + 10.0 * x1_pt  # 2(2)(3) + 30 = 42.0
grad = np.array([g0, g1])

# Second derivatives (Hessian Matrix)
H = np.array(
    [[6.0 * x0_pt, 2.0 * x1_pt], [2.0 * x1_pt, 2.0 * x0_pt + 10.0]]
)  # [[12, 6], [6, 14]]

# Eigenvalues & Eigenvectors of the Hessian
eigenvalues, eigenvectors = np.linalg.eigh(H)
# Both eigenvalues > 0 -> Upward-curving bowl (Positive Definite)

# 2. Base surface grid around (2.0, 3.0)
span = 1.0
x0_vals = np.linspace(x0_pt - span, x0_pt + span, 45)
x1_vals = np.linspace(x1_pt - span, x1_pt + span, 45)
X0, X1 = np.meshgrid(x0_vals, x1_vals)
Z_true = f(X0, X1)

# 3. 2nd-order Taylor Approximation Surface using Hessian:
# f_approx(x) = f(p) + grad · Δx + 0.5 * Δx^T H Δx
dX0 = X0 - x0_pt
dX1 = X1 - x1_pt
Z_quad = (
    z_pt
    + (g0 * dX0 + g1 * dX1)
    + 0.5 * (H[0, 0] * (dX0**2) + 2.0 * H[0, 1] * dX0 * dX1 + H[1, 1] * (dX1**2))
)

# 4. Curvature tracks along principal Hessian eigenvectors
t = np.linspace(-0.7, 0.7, 40)
# Axis of Maximum Curvature (eigenvector 1)
v_max = eigenvectors[:, 1]
c1_x0 = x0_pt + t * v_max[0]
c1_x1 = x1_pt + t * v_max[1]
c1_z = f(c1_x0, c1_x1)

# Axis of Minimum Curvature (eigenvector 0)
v_min = eigenvectors[:, 0]
c2_x0 = x0_pt + t * v_min[0]
c2_x1 = x1_pt + t * v_min[1]
c2_z = f(c2_x0, c2_x1)

# 5. Build Plotly figure
fig = go.Figure()

# True Surface
fig.add_trace(
    go.Surface(
        x=X0,
        y=X1,
        z=Z_true,
        colorscale="Blues",
        opacity=0.65,
        showscale=False,
        name="True Surface f(x)",
    )
)

# Quadratic Paraboloid (Hessian Approximation)
fig.add_trace(
    go.Surface(
        x=X0,
        y=X1,
        z=Z_quad,
        colorscale="YlOrRd",
        opacity=0.45,
        showscale=False,
        name="Hessian Quadratic Fit",
    )
)

# Principal Curvature Lines
fig.add_trace(
    go.Scatter3d(
        x=c1_x0,
        y=c1_x1,
        z=c1_z,
        mode="lines",
        line={"color": "#d62728", "width": 7},
        name=f"Max Curvature (λ₁ = {eigenvalues[1]:.1f})",
    )
)

fig.add_trace(
    go.Scatter3d(
        x=c2_x0,
        y=c2_x1,
        z=c2_z,
        mode="lines",
        line={"color": "#2ca02c", "width": 7},
        name=f"Min Curvature (λ₂ = {eigenvalues[0]:.1f})",
    )
)

# Evaluation Point
fig.add_trace(
    go.Scatter3d(
        x=[x0_pt],
        y=[x1_pt],
        z=[z_pt],
        mode="markers+text",
        marker={"size": 7, "color": "black"},
        text=[f"P({x0_pt}, {x1_pt}, {z_pt:.0f})"],
        textposition="top left",
        name="Point (2, 3)",
    )
)

fig.update_layout(
    title=f"Hessian Curvature at (2.0, 3.0)<br>"
    f"Positive Definite (λ₁ = {eigenvalues[1]:.2f}, λ₂ = {eigenvalues[0]:.2f}) ➔ Upward Bowl",
    scene={
        "xaxis_title": "x₀",
        "yaxis_title": "x₁",
        "zaxis_title": "f(x)",
        "aspectratio": {"x": 1, "y": 1, "z": 0.75},
        "camera": {"eye": {"x": -1.5, "y": -1.5, "z": 1.1}},
    },
    width=880,
    height=620,
    legend={"x": 0.02, "y": 0.95},
)

fig.show()

### Concept 7: Gradient Descent with Momentum

### What is Gradient Descent with Momentum? (Simple English)

Standard **Gradient Descent** moves like an ant: at every step, it looks solely at the local slope under its feet and takes a step downhill.

* If the landscape has a narrow ravine with steep walls and a gentle downhill floor, the ant bounces violently back and forth between the walls, making painfully slow progress down the valley.
* If it encounters a tiny bump, a flat plateau, or a saddle point ($\nabla f \approx 0$), it stalls and stops completely.

**Momentum changes the ant into a heavy rolling ball:**

> *"Instead of using the current gradient directly as your step, use the gradient to push a velocity vector with physical inertia."*

* When the ball rolls down a consistent hill, it gathers speed in that direction.
* In directions where the gradients oscillate (bouncing back and forth across walls), the opposing pushes cancel each other out.
* When the ball encounters a flat spot or minor local bump, its accumulated momentum carries it straight through.


### The Formulas

At iteration $t$, we maintain two quantities: current parameters $\mathbf{w}_t$ and a velocity vector $\mathbf{v}_t$.

**1. Velocity Update (Accumulating momentum):**

$$\mathbf{v}_{t+1} = \beta \mathbf{v}_t + \eta \nabla f(\mathbf{w}_t)$$

**2. Parameter Step (Moving along velocity):**

$$\mathbf{w}_{t+1} = \mathbf{w}_t - \mathbf{v}_{t+1}$$

#### How to read it out loud:

> *"The new velocity $v_{t+1}$ equals beta times the previous velocity $v_t$, plus the learning rate eta times the gradient at point $w_t$. The new parameters $w_{t+1}$ equal $w_t$ minus the updated velocity."*

#### Breakdown of every piece:

* **$\mathbf{w}_t$**: Current weights/parameters at step $t$.
* **$\nabla f(\mathbf{w}_t)$**: The gradient vector (the push/acceleration from the slope).
* **$\eta$** (*eta*): The **learning rate** (how hard the slope pushes the velocity).
* **$\beta$** (*beta*, typically $0.9$): The **momentum coefficient** (friction factor).
* If $\beta = 0$, velocity forgets past steps immediately and reduces to standard Gradient Descent.
* If $\beta = 0.9$, about $90\%$ of previous velocity is retained across steps, accelerating consistent directions up to a factor of $\frac{1}{1 - \beta} = 10\times$.


* **$\mathbf{v}_t$**: The velocity vector storing running inertia.

### Why AI Cares (In plain terms)

* **SGD with Momentum:** The default optimizer for training computer vision architectures (ResNets, ConvNets, YOLO).
* **The Backbone of Adam & Modern Optimizers:** Optimizers like Adam, AdamW, and RMSProp rely directly on momentum: Adam's first moment ($m_t$) is literally exponentially decaying momentum applied to mini-batch gradients.



Let's test both standard Gradient Descent and Momentum on an elongated ravine function (ill-conditioned quadratic bowl) where $x_0$ has very steep curvature and $x_1$ has gentle curvature:

$$f(x_0, x_1) = 10x_0^2 + x_1^2$$

The global minimum is at $(0.0, 0.0)$.

> Notice how Momentum damps oscillations along the steep axis ($x_0$) while speeding up traversal down the shallow axis ($x_1$), reaching a significantly lower loss within the same number of steps.

### NumPy Implementation

In [ ]:
# 1. Define ill-conditioned valley function and its analytic gradient
def loss_fn(w):
    return 10.0 * (w[0] ** 2) + (w[1] ** 2)


def grad_fn(w):
    return np.array([20.0 * w[0], 2.0 * w[1]])


# Initial starting point far from the origin
start_point = np.array([2.0, 10.0])
num_steps = 30
learning_rate = 0.08

# ==========================================
# Run Standard Gradient Descent (No Momentum)
# w = w - lr * grad
# ==========================================
w_gd = start_point.copy()
gd_history = [w_gd.copy()]

for _ in range(num_steps):
    g = grad_fn(w_gd)
    w_gd -= learning_rate * g
    gd_history.append(w_gd.copy())

# ==========================================
# Run Gradient Descent with Momentum
# v = beta * v + lr * grad
# w = w - v
# ==========================================
w_mom = start_point.copy()
v = np.zeros_like(w_mom)
beta = 0.85
mom_history = [w_mom.copy()]

for _ in range(num_steps):
    g = grad_fn(w_mom)
    # Update velocity
    v = beta * v + learning_rate * g
    # Update weights
    w_mom -= v
    mom_history.append(w_mom.copy())

print(f"Starting Point:       {start_point}")
print(f"Starting Loss:        {loss_fn(start_point):.4f}\n")

print(f"Final Point (Standard GD): {np.round(w_gd, 4)}")
print(f"Final Loss  (Standard GD): {loss_fn(w_gd):.6f}\n")

print(f"Final Point (With Momentum): {np.round(w_mom, 4)}")
print(f"Final Loss  (With Momentum): {loss_fn(w_mom):.6f}")

Starting Point:       [ 2. 10.]
Starting Loss:        140.0000

Final Point (Standard GD): [0.     0.0535]
Final Loss  (Standard GD): 0.002863

Final Point (With Momentum): [0.2151 0.8547]
Final Loss  (With Momentum): 1.193260


In [ ]:
# 1. Target function & gradients
def loss_fn(w0, w1):
    return 10.0 * (w0**2) + (w1**2)


def grad_fn(w):
    return np.array([20.0 * w[0], 2.0 * w[1]])


start_point = np.array([2.0, 10.0])
num_steps = 30
learning_rate = 0.08

# 2. Simulate Standard Gradient Descent
w_gd = start_point.copy()
gd_history = [w_gd.copy()]
for _ in range(num_steps):
    g = grad_fn(w_gd)
    w_gd = w_gd - learning_rate * g
    gd_history.append(w_gd.copy())
gd_history = np.array(gd_history)
gd_z = loss_fn(gd_history[:, 0], gd_history[:, 1])

# 3. Simulate Gradient Descent with Momentum (beta = 0.85)
w_mom = start_point.copy()
v = np.zeros_like(w_mom)
beta = 0.85
mom_history = [w_mom.copy()]
for _ in range(num_steps):
    g = grad_fn(w_mom)
    v = beta * v + learning_rate * g
    w_mom = w_mom - v
    mom_history.append(w_mom.copy())
mom_history = np.array(mom_history)
mom_z = loss_fn(mom_history[:, 0], mom_history[:, 1])

# 4. Generate 3D surface mesh
w0_grid = np.linspace(-3.0, 3.0, 50)
w1_grid = np.linspace(-2.0, 12.0, 50)
W0, W1 = np.meshgrid(w0_grid, w1_grid)
Z = loss_fn(W0, W1)

# 5. Build Plotly figure
fig = go.Figure()

# 3D Elliptical Bowl with floor contours
fig.add_trace(
    go.Surface(
        x=W0,
        y=W1,
        z=Z,
        colorscale="Blues",
        opacity=0.6,
        showscale=False,
        contours={
            "z": {"show": True, "project_z": True, "usecolormap": True, "highlightcolor": "gray"}
        },
        name="Loss Surface f(w)",
    )
)

# Path A: Standard Gradient Descent (Oscillating)
fig.add_trace(
    go.Scatter3d(
        x=gd_history[:, 0],
        y=gd_history[:, 1],
        z=gd_z,
        mode="lines+markers",
        marker={"size": 4, "color": "#d62728"},
        line={"color": "#d62728", "width": 5},
        name=f"Standard GD (Loss: {gd_z[-1]:.3f})",
    )
)

# Path B: Momentum (Damped & Accelerated)
fig.add_trace(
    go.Scatter3d(
        x=mom_history[:, 0],
        y=mom_history[:, 1],
        z=mom_z,
        mode="lines+markers",
        marker={"size": 4, "color": "#2ca02c"},
        line={"color": "#2ca02c", "width": 6},
        name=f"With Momentum β=0.85 (Loss: {mom_z[-1]:.4f})",
    )
)

# Start and Optimum Points
fig.add_trace(
    go.Scatter3d(
        x=[start_point[0]],
        y=[start_point[1]],
        z=[loss_fn(start_point[0], start_point[1])],
        mode="markers+text",
        marker={"size": 8, "color": "black"},
        text=["Start [2, 10]"],
        textposition="top right",
        name="Start Point",
    )
)

fig.add_trace(
    go.Scatter3d(
        x=[0.0],
        y=[0.0],
        z=[0.0],
        mode="markers+text",
        marker={"size": 7, "color": "#ff7f0e", "symbol": "diamond"},
        text=["Minimum (0, 0)"],
        textposition="bottom center",
        name="Global Minimum",
    )
)

fig.update_layout(
    title="Standard GD vs Momentum on Ill-Conditioned Valley<br>"
    f"Standard GD Loss: {gd_z[-1]:.4f} | Momentum Loss: {mom_z[-1]:.4f}",
    scene={
        "xaxis_title": "w₀ (Steep Curvature)",
        "yaxis_title": "w₁ (Gentle Curvature)",
        "zaxis_title": "Loss f(w)",
        "aspectratio": {"x": 1, "y": 1.2, "z": 0.75},
        "camera": {"eye": {"x": -1.6, "y": -1.4, "z": 1.2}},
    },
    width=900,
    height=650,
    legend={"x": 0.02, "y": 0.95},
)

fig.show()